<a href="https://colab.research.google.com/github/ldongheedev/-BDA-LLM-RAG-Program/blob/main/8%EC%A3%BC%EC%B0%A8_%EA%B3%BC%EC%A0%9C2_%ED%94%84%EB%A1%AC%ED%94%84%ED%8A%B8%EC%84%A4%EA%B3%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 8주차 과제 2: 프롬프트 설계

3가지 과제(감성 분류, 텍스트 요약, 키워드 추출)를 프롬프트로 해결합니다.
각 과제는 **1차 설계 → 테스트 → 개선 → 최종 테스트** 과정을 기록합니다.


In [ ]:
!pip install transformers


In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="skt/kogpt2-base-v2",
)
print("모델 로드 완료!")


---
## 1. 감성 분류 (긍정 / 부정 / 중립)

### 1차 설계: 단순 Zero-shot


In [ ]:
# 1차: 단순하게 물어보기
prompt_v1 = """다음 리뷰가 긍정인지 부정인지 알려줘.
리뷰: 가격 대비 만족합니다
답변:"""

result = generator(prompt_v1, max_length=len(prompt_v1.split()) + 15, do_sample=False)
print("[1차 설계 — Zero-shot]")
print(result[0]["generated_text"])
print()
print("문제점: 출력 형식 불명확, 중립 선택지 없음")


### 2차 설계: 역할 부여 + 출력 형식 지정


In [ ]:
# 2차: 역할 부여 + 형식 지정
prompt_v2 = """당신은 상품 리뷰 감성 분석 전문가입니다.
리뷰를 읽고 긍정/부정/중립 중 하나로 분류하세요.
답변은 분류 결과 한 단어만 출력하세요.

리뷰: 가격 대비 만족합니다
감성:"""

result = generator(prompt_v2, max_length=len(prompt_v2.split()) + 10, do_sample=False)
print("[2차 설계 — 역할 부여 + 형식 지정]")
print(result[0]["generated_text"])
print()
print("개선: 역할 부여로 답변 품질 향상, 한 단어 제약으로 깔끔")
print("문제: 애매한 케이스의 판단 기준 부족")


### 3차 설계 (최종): Few-shot + 판단 기준 명시


In [ ]:
# 3차(최종): Few-shot + 판단 기준
prompt_v3 = """당신은 상품 리뷰 감성 분석 전문가입니다.
리뷰를 읽고 긍정/부정/중립 중 하나로 분류하세요.

판단 기준:
- 긍정: 만족, 추천, 좋음을 표현
- 부정: 불만, 실망, 나쁨을 표현
- 중립: 장단점이 섞여있거나 객관적 사실만 서술

예시:
리뷰: 품질 좋고 배송 빨라요 → 긍정
리뷰: 불량이고 환불 어렵다 → 부정
리뷰: 디자인은 좋은데 내구성이 약해요 → 중립

리뷰: 가격 대비 만족합니다 →"""

result = generator(prompt_v3, max_length=len(prompt_v3.split()) + 10, do_sample=False)
print("[3차 설계 — Few-shot + 판단 기준]")
print(result[0]["generated_text"][-80:])
print()
print("KoGPT2는 소규모 모델이라 정확도가 제한적입니다.")
print("GPT-4/Claude에서는 이 프롬프트로 정확하게 '긍정'을 출력합니다.")


In [ ]:
# 최종 프롬프트로 여러 리뷰 테스트
test_reviews = [
    "배송 빠르고 포장 꼼꼼해요",
    "색상이 사진과 달라서 실망했어요",
    "가격은 괜찮은데 소재가 좀 아쉽네요",
    "이 가격에 이 품질이면 최고입니다",
    "한 달 만에 고장났어요 환불 원합니다",
]

template = """리뷰: 품질 좋고 배송 빨라요 → 긍정
리뷰: 불량이고 환불 어렵다 → 부정
리뷰: 디자인은 좋은데 내구성이 약해요 → 중립
리뷰: {} →"""

print("=== 감성 분류 테스트 ===")
for review in test_reviews:
    prompt = template.format(review)
    result = generator(prompt, max_length=len(prompt.split()) + 5, do_sample=False)
    generated = result[0]["generated_text"]
    # 마지막 → 이후 부분 추출
    answer = generated.split("→")[-1].strip()[:10]
    print(f"  '{review}' → {answer}")


---
## 2. 텍스트 요약 (3줄)

### 1차 설계: 단순 Zero-shot


In [ ]:
# 요약할 텍스트
article = """인공지능(AI)은 컴퓨터 시스템이 인간의 지능적 행동을 모방하는 기술이다. \
최근 딥러닝과 대규모 언어 모델의 발전으로 AI는 텍스트 생성, 이미지 인식, \
자연어 처리 등 다양한 분야에서 혁신적인 성과를 보이고 있다. \
특히 ChatGPT와 같은 대화형 AI는 일상생활과 업무 환경에서 \
널리 활용되고 있으며, 교육, 의료, 법률 등 전문 분야에서도 \
활용 범위가 확대되고 있다. 그러나 AI의 편향성, 저작권 문제, \
일자리 대체 우려 등 사회적 과제도 함께 논의되고 있다."""

# 1차: 단순하게
prompt_v1 = f"""다음 글을 요약해줘.

{article}

요약:"""

result = generator(prompt_v1, max_length=len(prompt_v1.split()) + 50, do_sample=True, temperature=0.5)
print("[1차 설계 — 단순 Zero-shot]")
print(result[0]["generated_text"].split("요약:")[-1].strip()[:200])
print()
print("문제점: 요약 길이, 형식이 불명확")


### 2차 설계 (최종): 줄 수 + 형식 지정


In [ ]:
# 2차(최종): 구체적 형식 지정
prompt_v2 = f"""다음 텍스트를 정확히 3줄로 요약하세요.
각 줄은 한 문장으로 핵심 내용만 담아주세요.
번호를 붙여서 작성하세요.

텍스트:
{article}

요약:
1."""

result = generator(prompt_v2, max_length=len(prompt_v2.split()) + 60, do_sample=True, temperature=0.5)
print("[2차 설계 — 줄 수 + 형식 지정]")
output = result[0]["generated_text"].split("요약:")[-1].strip()
print(output[:300])
print()
print("KoGPT2는 요약 능력이 제한적입니다.")
print("GPT-4/Claude에서 이 프롬프트를 사용하면 깔끔한 3줄 요약을 얻을 수 있습니다.")
print()
print("GPT-4/Claude 예상 결과:")
print("  1. AI는 딥러닝과 대규모 언어 모델의 발전으로 다양한 분야에서 혁신적 성과를 보이고 있다.")
print("  2. ChatGPT 같은 대화형 AI가 일상과 전문 분야에서 널리 활용되고 있다.")
print("  3. 편향성, 저작권, 일자리 대체 등 사회적 과제도 함께 논의되고 있다.")


In [ ]:
# 다른 텍스트로 요약 테스트
article2 = """최근 전기차 시장이 급성장하고 있다. \
테슬라를 비롯한 글로벌 자동차 업체들이 전기차 생산을 확대하고 있으며, \
각국 정부도 탄소 중립 목표 달성을 위해 전기차 보조금을 지급하고 있다. \
배터리 기술의 발전으로 주행 거리가 늘어나고 충전 시간도 단축되고 있다. \
하지만 충전 인프라 부족, 배터리 원자재 수급 문제, \
폐배터리 처리 등 해결해야 할 과제도 남아있다."""

prompt = f"""다음 텍스트를 정확히 3줄로 요약하세요.
각 줄은 한 문장으로 핵심 내용만 담아주세요.
번호를 붙여서 작성하세요.

텍스트:
{article2}

요약:
1."""

result = generator(prompt, max_length=len(prompt.split()) + 60, do_sample=True, temperature=0.5)
output = result[0]["generated_text"].split("요약:")[-1].strip()
print("[전기차 기사 요약 테스트]")
print(output[:300])


---
## 3. 키워드 추출 (5개)

### 1차 설계: 단순 Zero-shot


In [ ]:
# 1차: 단순하게
prompt_v1 = f"""다음 글에서 키워드를 뽑아줘.

{article}

키워드:"""

result = generator(prompt_v1, max_length=len(prompt_v1.split()) + 20, do_sample=False)
output = result[0]["generated_text"].split("키워드:")[-1].strip()
print("[1차 설계 — 단순 Zero-shot]")
print(output[:150])
print()
print("문제점: 키워드 개수 불명확, 출력 형식 제각각")


### 2차 설계 (최종): 개수 + 형식 + 제약


In [ ]:
# 2차(최종): 구체적 조건 지정
prompt_v2 = f"""다음 텍스트에서 가장 중요한 핵심 키워드 5개를 추출하세요.
중요도 순으로 쉼표로 구분하여 한 줄로 나열하세요.
키워드만 출력하고 다른 설명은 하지 마세요.

텍스트:
{article}

키워드:"""

result = generator(prompt_v2, max_length=len(prompt_v2.split()) + 20, do_sample=False)
output = result[0]["generated_text"].split("키워드:")[-1].strip()
print("[2차 설계 — 개수 + 형식 + 제약]")
print(output[:150])
print()
print("KoGPT2는 지시 따르기(instruction following) 능력이 제한적입니다.")
print("GPT-4/Claude 예상 결과: 인공지능, 딥러닝, 대규모 언어 모델, 텍스트 생성, 자연어 처리")


In [ ]:
# 다른 텍스트로 키워드 추출 테스트
prompt = f"""다음 텍스트에서 가장 중요한 핵심 키워드 5개를 추출하세요.
중요도 순으로 쉼표로 구분하여 한 줄로 나열하세요.
키워드만 출력하고 다른 설명은 하지 마세요.

텍스트:
{article2}

키워드:"""

result = generator(prompt, max_length=len(prompt.split()) + 20, do_sample=False)
output = result[0]["generated_text"].split("키워드:")[-1].strip()
print("[전기차 기사 키워드 추출 테스트]")
print(output[:150])
print()
print("GPT-4/Claude 예상 결과: 전기차, 배터리, 탄소 중립, 충전 인프라, 보조금")


---
## 4. KoGPT2 vs 대형 모델 — Few-shot 성능 비교


In [ ]:
# Few-shot 감성 분류를 KoGPT2에서 테스트
prompt = """다음은 영화 리뷰의 감성 분류 예시입니다.

리뷰: 정말 재미있는 영화였다
감성: 긍정

리뷰: 시간 낭비였다
감성: 부정

리뷰: 배우 연기가 좋았다
감성:"""

result = generator(prompt, max_length=len(prompt.split()) + 10, do_sample=False)
print("=== Few-shot 감성 분류 (KoGPT2) ===")
print(result[0]["generated_text"][-60:])
print()
print("─" * 50)
print()
print("GPT-4/Claude에서는 이 프롬프트로 정확하게 '긍정'을 출력합니다.")
print("직접 ChatGPT(chat.openai.com) 또는 Claude(claude.ai)에")
print("위 프롬프트를 붙여넣어 테스트해보세요!")


In [ ]:
# Few-shot 키워드 추출을 KoGPT2에서 테스트
prompt = """다음은 텍스트에서 키워드를 추출하는 예시입니다.

텍스트: 서울의 지하철은 편리하고 요금이 저렴하다
키워드: 서울, 지하철, 편리, 요금, 저렴

텍스트: 최근 반도체 수출이 증가하면서 경제 성장률이 높아졌다
키워드: 반도체, 수출, 증가, 경제, 성장률

텍스트: 올해 여름은 폭염과 집중호우가 번갈아 나타났다
키워드:"""

result = generator(prompt, max_length=len(prompt.split()) + 15, do_sample=False)
print("=== Few-shot 키워드 추출 (KoGPT2) ===")
print(result[0]["generated_text"][-60:])
print()
print("GPT-4/Claude 예상 결과: 여름, 폭염, 집중호우, 올해, 번갈아")


---
## 프롬프트 설계 과정 정리

### 각 과제별 개선 과정

| 과제 | 1차 (문제점) | 최종 (개선) |
|------|-------------|------------|
| 감성 분류 | "긍정인지 부정인지 알려줘" (형식 불명확) | 역할 부여 + 판단 기준 + Few-shot 예시 3개 |
| 텍스트 요약 | "요약해줘" (길이/형식 불명확) | "정확히 3줄" + "한 문장씩" + "번호 붙여서" |
| 키워드 추출 | "키워드 뽑아줘" (개수/형식 불명확) | "5개" + "중요도 순" + "쉼표 구분" + "키워드만 출력" |

### 좋은 프롬프트의 공통 패턴

```
1. 역할 부여       → "당신은 ~전문가입니다"
2. 명확한 출력 형식 → "3줄로", "쉼표로 구분", "한 단어로만"
3. 예시 제공        → Few-shot으로 패턴 제시
4. 제약 조건        → "다른 설명 없이", "~만 출력"
```

### KoGPT2 vs 대형 모델

| 항목 | KoGPT2 (실습) | GPT-4 / Claude |
|------|--------------|----------------|
| 파라미터 | 약 1.25억 | 수천억 이상 |
| Zero-shot | 거의 불가 | 대부분 정확 |
| Few-shot | 제한적 | 매우 정확 |
| 지시 따르기 | 약함 | 강함 |

같은 Transformer 구조라도 **규모**를 키우고 **RLHF**로 정렬하면 프롬프트 성능이 극적으로 향상됩니다.
